# GX13 — Autovalores: exatos vs aproximação de Beck (1992)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ana-mat-br/codigos-livro-conducao-calor/blob/main/gx13_autovalores.ipynb)

**Livro:** *Condução de Calor: Aplicações das Funções de Green em Problemas de Engenharia*  
**Autores:** Ana Paula Fernandes e Gilmar Guimarães

---

## Descrição

O problema GX13 corresponde à placa plana com temperatura prescrita em $x=0$ (tipo 1) e convecção em $x=L$ (tipo 3). Os autovalores $\beta_m$ satisfazem a equação transcendental (Eq. 4.21):

$$\xi_m \cot(\xi_m) = -H_2, \qquad \xi_m = \beta_m L, \quad H_2 = \frac{h_2 L}{k}$$

Este notebook compara os autovalores **exatos** (obtidos pelo método de Brent) com os valores **aproximados** pelas fórmulas de Beck (1992), com e sem a correção iterativa de Newton-Raphson (Eq. 4.26).

## Bibliotecas

In [ ]:
import numpy as np
from scipy.optimize import brentq
import pandas as pd

## Parâmetro $H_2$

Altere o valor de `H2` para explorar diferentes regimes de convecção. As fórmulas de Beck cobrem:

| Fórmula | Validade | Equação no livro |
|---|---|---|
| Eq. (4.22) | $m=1$, $-1 \leq H_2 \leq -0{,}6$ | casos com $h_2 < 0$ |
| Eq. (4.23) | $m=1,2,\ldots$, $-1 < H_2 < 5$ | maioria dos problemas práticos |
| Eqs. (4.24)–(4.25) | $H_2 > 5$ | convecção intensa |

Em todos os casos, a **Eq. (4.26)** aplica uma correção iterativa de Newton-Raphson para aumentar a precisão.

In [ ]:
H2 = 2.0   # H2 = h2·L/k  — altere aqui para explorar diferentes casos
M  = 5     # número de autovalores

print(f'H2 = {H2:.4f}')

## Autovalores exatos — método de Brent

A equação $\xi\cot(\xi) = -H_2$ é equivalente a:

$$f(\xi) = \frac{\xi}{\tan(\xi)} + H_2 = 0$$

A função $\xi\cot(\xi)$ tem singularidades em $\xi = m\pi$ e vale zero em $\xi = (m-\tfrac{1}{2})\pi$. Para $H_2 > 0$, cada raiz $\xi_m$ está no intervalo $\bigl((m-1)\pi + \varepsilon,\; m\pi - \varepsilon\bigr)$:

| $m$ | Intervalo de busca |
|---|---|
| 1 | $(\varepsilon,\; \pi - \varepsilon)$ |
| 2 | $(\pi + \varepsilon,\; 2\pi - \varepsilon)$ |
| 3 | $(2\pi + \varepsilon,\; 3\pi - \varepsilon)$ |

O método de Brent garante encontrar a raiz em cada intervalo.

In [ ]:
eps = 1e-10

def f_gx13(xi, H2):
    """Equação transcendental GX13: ξ·cot(ξ) + H2 = 0"""
    return xi / np.tan(xi) + H2

xi_exato = np.array([
    brentq(f_gx13, (m-1)*np.pi + eps, m*np.pi - eps, args=(H2,))
    for m in range(1, M+1)
])

print('Autovalores exatos ξ_m = β_m · L:')
for m, xi in enumerate(xi_exato, 1):
    print(f'  ξ_{m} = {xi:.8f}')

## Aproximações de Beck (1992)

### Eq. (4.22) — $m=1$, $-1 \leq H_2 \leq -0{,}6$

$$\xi_1 = \left[3\left(1+H_2 - \tfrac{1}{5}(1+H_2)^2\right)\right]^{1/2}$$

### Eq. (4.23) — $m=1,2,\ldots$, $-1 < H_2 < 5$

$$\xi_m = \frac{\pi}{2}(2m-1)\left[1 + \frac{3}{2(H_2+3)}\left(\left(1 + \frac{16(H_2+3)}{3(2m-1)^2\pi^2}\right)^{\!1/2} - 1\right)\right]$$

### Eqs. (4.24)–(4.25) — $H_2 > 5$

$$A = \left[\left(\frac{3m\pi}{2H_2}\right)^2 + \left(1+\frac{1}{H_2}\right)^3\right]^{1/2}, \qquad
\xi_m = m\pi - \left(A + \frac{3m\pi}{2H_2}\right)^{1/3} + \left(A - \frac{3m\pi}{2H_2}\right)^{1/3}$$

### Eq. (4.26) — Correção de Newton-Raphson (aplicada iterativamente)

Partindo do valor aproximado $\xi'_m$, aplica-se repetidamente:

$$\xi_m \leftarrow \xi_m - \frac{H_2\tan\xi_m + \xi_m}{H_2/\cos^2\!\xi_m + 1}$$

até que a correção seja inferior a um critério de convergência $\psi$. Cada iteração resolve o sistema $F(\xi) = H_2\tan\xi + \xi = 0$, que é equivalente à Eq. (4.21).

In [ ]:
def beck_xi(m, H2):
    """Aproximação inicial de Beck para ξ_m de GX13."""
    if m == 1 and -1.0 <= H2 <= -0.6:          # Eq. (4.22)
        return np.sqrt(3 * (1 + H2 - (1 + H2)**2 / 5))
    elif H2 <= 5:                               # Eq. (4.23)
        n = 2*m - 1
        inner = (1 + 16*(H2+3) / (3*n**2*np.pi**2))**0.5 - 1
        return (np.pi/2) * n * (1 + 3/(2*(H2+3)) * inner)
    else:                                       # Eqs. (4.24)–(4.25)
        A = np.sqrt((3*m*np.pi / (2*H2))**2 + (1 + 1/H2)**3)
        t1 = (A + 3*m*np.pi / (2*H2))**(1/3)
        t2 = (A - 3*m*np.pi / (2*H2))**(1/3)
        return m*np.pi - t1 + t2

def newton_iter(xi0, H2, tol=1e-12, nmax=100):
    """Newton-Raphson iterativo — Eq. (4.26)."""
    xi = xi0
    for _ in range(nmax):
        F  = H2 * np.tan(xi) + xi
        Fp = H2 / np.cos(xi)**2 + 1
        dxi = F / Fp
        xi -= dxi
        if abs(dxi) < tol:
            return xi
    return xi   # retorna o melhor valor encontrado

xi_beck    = np.array([beck_xi(m, H2)                     for m in range(1, M+1)])
xi_beck_nr = np.array([newton_iter(beck_xi(m, H2), H2)    for m in range(1, M+1)])

## Tabela de comparação

A tabela abaixo mostra os autovalores exatos $\xi_m^{\text{exato}}$ (Brent), a aproximação inicial de Beck $\xi_m^{\text{Beck}}$, a aproximação com NR iterativo $\xi_m^{\text{Beck+NR}}$ e os erros relativos percentuais.

In [ ]:
erro_beck    = 100 * np.abs(xi_beck    - xi_exato) / xi_exato
erro_beck_nr = 100 * np.abs(xi_beck_nr - xi_exato) / xi_exato

df = pd.DataFrame({
    'm'               : range(1, M+1),
    'ξ_exato'         : xi_exato,
    'ξ_Beck'          : xi_beck,
    'Erro Beck (%)'   : erro_beck,
    'ξ_Beck+NR'       : xi_beck_nr,
    'Erro+NR (%)'     : erro_beck_nr,
})
df = df.set_index('m')

pd.set_option('display.float_format', '{:.6f}'.format)
print(f'H2 = {H2:.4f}\n')
print(df.to_string())

## Observações

- Para **$H_2 \geq 1$**: a aproximação de Beck (Eq. 4.23) fornece um valor inicial razoável, e o NR iterativo converge à precisão de máquina em poucas iterações.
- Para **$H_2 > 5$**: as Eqs. (4.24)–(4.25) são muito precisas já como aproximação inicial (erro < 0,01%), e o NR converge imediatamente.
- Para **$H_2 < 1$**: a aproximação inicial de Beck pode ser insuficiente para que o NR convirja. Nesse caso, recomenda-se usar diretamente o método de Brent (`brentq`) como solução exata.